# Actividad 2

## Detalles del equipo

### Nombre
**Panditas Panaderos**

### Integrantes
* Emilio Eduardo Garza RIvera - 2976048
* Adrián Alonso Jair Morales Márquez - 3110244
* Miguel Omar Muñoz Solis - 2953688
* Artur Emmanuel Martínez Martínez - 3098530
* Leonardo Yahir Martínez Estrada - 07000080

### Slogan
*A hornear*

# Información General del Dataset

**Nombre:** *Student Performance Factors Dataset*

**Fuente y liga de acceso:** *https://www.kaggle.com/datasets/mosapabdelghany/student-performance-factors-dataset*

**Fecha de consulta:** *13 de agosto de 2026 — 17 de agosto de 2026*

**Fecha de última actualización en Kaggle:** *Hace 10 meses*

**Descripción:** *Este conjunto contiene variables numéricas y categóricas relacionadas con factores académicos, personales, familiares y esoclares que pueden influir en el desempeño de los estudiantes. En esta actividad se tendrá como objetivo seleccionar las variables más importantes y entrenar los modelos para que puedan ser capaces de predecir cuáles variables son las que verdaderamente influyen.*

# Datos del Dataset

## Explicación de los registros

En este caso se usará el dataset limpio que se obtuvo de la actividad 1. Éstos son los registros originales y nuevos.

**Registros originales**
* **Hours_Studied:** *Número de horas de estudio por semana*
* **Attendance:** *Porcentaje de asistencia en el periodo*
* **Parental_Involvment:** *Nivel de apoyo académico por parte de los padres*
* **Access_to_Resources:** *Disponibilidad de recursos académicos como libros o internet*
* **Extracurricular_Activities:** *Si el estudiante está involucrado en actividades extracurriculares*
* **Sleep_Hours:** *Promedio de horas de sueño al día*
* **Previous_Scores:** *Promedio de calificaciones académicas previas*
* **Motivation_Level:** *Nivel de motivación del estudiante*
* **Internet_Access:** *Si el estudiante tiene acceso al internet*
* **Tutoring_Sessions:** *Número de sesiones de tutoría a la semana*
* **Family_Income:** *Nivel de ingresos familiares*
* **Teacher_Quality:** *Calidad general percibida de los docentes*
* **School_Type:** *Tipo de escuela (Pública o Privada)*
* **Peer_Influence:** *Influencia de los compañeros en el rendimiento académico*
* **Physical_Activity:** *Horas de actividad física a la semana*
* **Learning_Disabilities:** *Si el estudiante presenta dificultades de aprendizaje*
* **Parental_Education_Level:** *Nivel máximo de estudios de los padres*
* **Distance_from_Home:** *Distancia entre el hogar y la escuela (Cerca, Moderada, Lejos)*
* **Gender:** *Género del estudiante*
* **Exam_Score:** *Calificación final del examen*

### Variable objetivo
La variable **Exam_Score** es nuestra variable objetivo.

### Pregunta predictiva que guiará el análisis
¿Qué variables son las fundamentales para que un estudiante pueda obtener un puntaje alto en su examen? ¿Existe una combinación de variables que aumente o disminuya este puntaje?

# Paso 1 y 2

In [ ]:
%pip install matplotlib seaborn pandas scikit-learn kagglehub

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import kagglehub
from kagglehub import KaggleDatasetAdapter

### Limpieza que se realizó en la Actividad 1

In [ ]:
def clean_data():
    # Load the dataset from Kaggle
    file_path = "StudentPerformanceFactors.csv"
    
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "mosapabdelghany/student-performance-factors-dataset",
        file_path,
    )
    
    # Columns of type string with missing values
    df_objects = df.select_dtypes(include=['object'])
    missing_columns = df_objects.columns[df_objects.isnull().any()].tolist()
    
    # Columns of type int64 with missing values
    #df_int64 = df.select_dtypes(include=['int64'])
    #missing_columns += df_int64.columns[df_int64.isnull().any()].tolist()
    
    # Print the number of missing values for each column with missing values
    for column in missing_columns:
        print(f"Valores faltantes en '{column}': {df[column].isnull().sum()}")
    
    # Fill missing values with the mode of the respective columns
    for column in missing_columns:
        df[column] = df[column].fillna(df[column].mode()[0])
        
    # Convert hierarchical categorical variables to numerical using one-hot encoding
    map_level = {
      'Low': 0,
      'Medium': 1,
      'High': 2
    }
    
    hc_columns = ['Parental_Involvement', 'Motivation_Level']
    
    for column in hc_columns:
        df[f'{column}_encoded'] = df[column].map(map_level)
        
    # Convert categorical variables to numerical using one-hot encoding
    df = pd.get_dummies(df, columns=['School_Type', 'Internet_Access'], drop_first=True, dtype=int)
    
    # Clean atypical values in Exam_Score column
    over_100 = (df['Exam_Score'] > 100).sum()
    print(f"\nRegistros con calificación mayor a 100 en 'Exam_Score': {over_100}")
    
    under_100 = (df['Exam_Score'] < 0).sum()
    print(f"Registros con calificación menor a 0 en 'Exam_Score': {under_100}")
    
    df.loc[df['Exam_Score'] > 100, 'Exam_Score'] = 100
    df.loc[df['Exam_Score'] < 0, 'Exam_Score'] = 0
    
    print("\nValor mínimo y máximo de 'Exam_Score' después de la limpieza:")
    print(f"Valor mínimo: {df['Exam_Score'].min()}")
    print(f"Valor máximo: {df['Exam_Score'].max()}")
    
    # Standard scaling of 'Hours_Studied' and 'Attendance'
    cols_to_scale = ['Hours_Studied', 'Attendance']
    scaler = StandardScaler()
    
    for column in cols_to_scale:
        df[f'{column}_scaled'] = scaler.fit_transform(df[[column]])
        
    return df

df = clean_data()

print("Descripción estadística del DataFrame:")
print(f"\nNúmero total de registros: {len(df)}")
print(f"\nNúmero de columnas: {len(df.columns)}")

print("\nValidación de cero valores nulos en el DataFrame:")
print(f"Valores nulos en el DataFrame: {df.isnull().sum().sum()}")

print("\nTipos de datos de las columnas después de la limpieza:")
print(df.dtypes)

print("\nNúmero de registros duplicados en el DataFrame después de la limpieza:")
print(df.duplicated().sum())

df.describe(include="all")

# Conclusión y Justificación de Cambios

**Registros nuevos**
* **Parental_Involvement_Encoded:** *Nivel de apoyo académico por parte de los padres medido en valores númericos (del 0 al 2)*
* **Motivation_Level_Encoded:** *Niveo de motivación del estudiante medido en valores númericos (del 0 al 2)*
* **School_Type_Private:** *Si el estudiante atiende escuela privada (0 siendo Falso y 1 siendo Verdadero)* *
* **School_Type_Public:** *Si el estudiante atiende escuela pública (0 siendo Falso y 1 siendo Verdadero)* 
* **Internet_Access_No:** *Si el estudiante no posee acceso a internet (0 siendo Falso y 1 siendo Verdadero)* *
* **Internet_Access_Yes:** *Si el estudiante sí posee acceso a internet (0 siendo Falso y 1 siendo Verdadero)*
* **Hours_Studied_Scaled:** *Número de horas de estudio por semana estandarizadas. Centrado en 0 con una desviación estándar de 1.*
* **Attendance_Scaled:** *Porcentaje de asistencia en el periodo estandarizado. Centrado en 0 con una desviación estándar de 1.*

Para el caso de la Actividad 1, se habían dejado todos las columnas anteriores como una forma de demostrar las transformaciones hechas. Para esta Actividad 2, las únicas modificaciones que se realizaron fue no incluir las columnas **School_Type_Private** y **Internet_Access_No** debido a que proveen la misma información que sus respectivas columnas **School_Type_Public** y **Internet_Access_Yes**.

Las demás transformaciones ayudan a tomar en cuenta los valores categóricos y no numéricos para poder relacionarlos con las variables numéricas y con la variable objetivo. De esta manera, por ejemplo en la matriz de correlación que se hará en el siguiente paso, se puede trabajar más fácilmente con todas las variables para determinar su nivel de influencia sobre la variable objetivo.

# Paso 3

Explorando la variable 'Exam_Score'

In [ ]:
print("\nEstadística descriptiva de la columna 'Exam_Score':")
df['Exam_Score'].describe()

# Histogram of 'Exam_Score'
plt.figure(figsize=(10, 6))
sns.histplot(df['Exam_Score'], bins=20, kde=True, color='blue')
plt.title('Distribución de Calificaciones de Examen')
plt.xlabel('Calificación de Examen')
plt.ylabel('Frecuencia')
plt.show()

# Boxplot of 'Exam_Score'
plt.figure(figsize=(10, 6))
sns.boxplot(x=df['Exam_Score'], color='lightgreen')
plt.title('Boxplot de Calificaciones de Examen')
plt.xlabel('Calificación de Examen')
plt.show()

In [ ]:
# Correlation heatmap of the numerical columns
plt.figure(figsize=(12, 8))
numerical_columns = df.select_dtypes(include=['number']).columns
correlation_matrix = df[numerical_columns].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Matriz de Correlación')
plt.show()

In [ ]:
# Diagrama de dispersión de las variables numéricas
plt.figure(figsize=(20, 8))
sns.pairplot(df[numerical_columns], diag_kind='kde', plot_kws={'alpha': 0.5})
plt.suptitle('Diagramas de Dispersión de Variables Numéricas', y=1.02)
plt.show() 

# Vairables que se seleccionarán

A través de la matriz de correlación y de los diagramas de dispersión, las variables que se seleccionarán en este caso son **Hours_Studied** y **Attendance**. Estas dos variables son las que mayor nivel de correlación tienen, con valores de **0.45** y **0.58** respectivamente.
La razón por la que éstas podrían cotribuir a explicar o predecir el puntaje del examen es debido a que, en el contexto académico, demuestran una consistencia y determinación por parte de los estudiantes. Tiene sentido que, si un estudiante pasa horas estudiando y va a clases todos los días, de alguna manera haya aprendido algo para el momento en el que toma el examen. Claro está que esto es sólo una hipótesis empírica personal/compartida, pues sabemos que hay casos donde esto no es la norma. Para ello, exploraremos y entrenaremos los modelos capaces de predecir la variable **Exam_Score** en esta actividad y determinar qué tan correcta es nuestra hipótesis.

# Paso 4

In [ ]:
from sklearn.model_selection import train_test_split

# Definimos la variable objetivo y las variables dependientes.
x = df[['Attendance', 'Hours_Studied']] 
y = df['Exam_Score']

#Dividimos el dataset en conjuntos de entrenamiento y prueba (80% / 20%) con una semilla fija de 42 para asegurar un resultado reproducible.
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=42)

Las computadoras no son capaces de generar números completamente al azar, para poder hacerlo utilizan algo llamado generador pseudoaleatorio, el cual es un algoritmo matemático que crea una secuencia de valores que parecen aleatorios a partir de un proceso determinista.

Este generador pseudoaleatorio necesita un valor inicial para poder iniciar a generar la secuencia de números, ese valor inicial se le conoce como semilla.

Esta semilla permite la reproducibilidad de los datos, ya que al usar el mismo valor el generador producirá exactamente la misma cadena de valores.

En este caso nos permite mantener constantes los valores aleatorios de nuestros modelos, lo que nos facilita el poder encontrar errores al evitar que los datos cambien en cada prueba.

En este caso utilizamos el número 42 como semilla fija de nuestros modelos, siguiendo el estándar que surgió a partir de un chiste interno de los primeros desarrolladores de algunas librerías de Python que lo utilizaban en sus códigos de ejemplo, como un homenaje a la novela The Hitchhiker's Guide to the Galaxy.

En el libro, una raza de seres muy inteligentes creó una máquina llamada Deep Thought a la que le preguntaron "¿Cuál es el sentido de la vida, el universo y todo lo demás?", a lo que la computadora, después de 7.5 millones de años, respondió con "42".

In [ ]:
print(f"Tamaño de entrenamiento (x_train): {x_train.shape}")
print(f"Tamaño de prueba (x_test): {x_test.shape}")

# Paso 5

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Definimos las columnas numericas de las variables seleccionadas.
numeric_features = ['Attendance', 'Hours_Studied']

# Pipeline numerico.
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features)
    ])


# Paso 6

In [ ]:
from sklearn.dummy import DummyRegressor

# Construimos y entrenamos el modelo base usando la media
dummy_regr = DummyRegressor(strategy="mean")
dummy_regr.fit(x_train, y_train)

# Generamos predicciones base
y_pred_dummy = dummy_regr.predict(x_test)

In [ ]:
print(f"Valor promedio obtenido del modelo: {y_pred_dummy[0]:.2f}")
print(f"Primeras 5 predicciones: {y_pred_dummy[:5]}")

# Paso 7

In [ ]:
from sklearn.linear_model import LinearRegression

# Modelo de Regresión Lineal
pipeline_lineal = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Entrenamiento del pipeline
pipeline_lineal.fit(x_train, y_train)

# Predicciones sobre el conjunto de prueba
y_pred_lineal = pipeline_lineal.predict(x_test)

In [ ]:
print("Primeras 5 predicciones (Regresión Lineal):")
print(y_pred_lineal[:5])

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Modelo de Árbol de Regresión
pipeline_arbol = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', DecisionTreeRegressor(random_state=42, max_depth=5)) # max_depth=5 para evitar sobreajuste
])

# Entrenamiento del pipeline
pipeline_arbol.fit(x_train, y_train)

# Predicciones sobre el conjunto de prueba
y_pred_arbol = pipeline_arbol.predict(x_test)



In [ ]:
print("Primeras 5 predicciones (Árbol de Regresión):")
print(y_pred_arbol[:5])

# Paso 8

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Función para calcular las tres métricas
def calcular_metricas(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return round(mae, 4), round(rmse, 4), round(r2, 4)

# Calculamos las métricas para los 3 modelos
mae_base, rmse_base, r2_base = calcular_metricas(y_test, y_pred_dummy)
mae_lin, rmse_lin, r2_lin = calcular_metricas(y_test, y_pred_lineal)
mae_arbol, rmse_arbol, r2_arbol = calcular_metricas(y_test, y_pred_arbol)

# Construimos la tabla de resultados
tabla_resultados = {
    'Modelo': ['Línea base', 'Regresión lineal', 'Modelo alternativo (Árbol)'],
    'MAE': [mae_base, mae_lin, mae_arbol],
    'RMSE': [rmse_base, rmse_lin, rmse_arbol],
    'R²': [r2_base, r2_lin, r2_arbol]
}

df_resultados = pd.DataFrame(tabla_resultados)

In [ ]:
# Mostramos la tabla
print("Tabla de Resultados:")
display(df_resultados)

# Paso 9

Interpretación de Métricas de Evaluación 

Al evaluar el rendimiento de los modelos frente a la línea base, la Regresión Lineal demostró ser la mejor opción para este conjunto de datos.

Su Error Absoluto Medio (MAE) fue de 1.4690 puntos. Esto representa que, en promedio, las predicciones del modelo se desvían por apenas un punto y medio respecto a la calificación real del estudiante. Considerando una escala de evaluación típica (por ejemplo, de 0 a 100), un margen de error tan pequeño indica que el modelo tiene una gran utilidad práctica. Nos permite estimar el desempeño académico de un alumno de manera bastante confiable, ayudando a identificar a tiempo si un estudiante requiere más apoyo.

Por otro lado, el RMSE obtenido por la regresión lineal fue de 2.4102. Como esta métrica penaliza con mayor intensidad los errores grandes al elevarlos al cuadrado antes de promediarlos, el hecho de que sea superior al MAE nos revela información importante: existen casos con errores de estimación más amplios. En el contexto escolar, esto es completamente natural, ya que siempre habrá estudiantes "atípicos" cuyos resultados rompen la tendencia (por ejemplo, alguien que estudia muy pocas horas pero obtiene una calificación excelente por conocimientos previos, o viceversa).

Finalmente, el coeficiente $R^2$ de 0.5890 aporta claridad sobre el nivel de ajuste del modelo. Esto significa que casi el 59% de la variabilidad en las calificaciones de los exámenes (Exam_Score) puede explicarse utilizando únicamente dos variables: la Asistencia y las Horas de Estudio. El ~41% restante obedece a factores que no introdujimos en este modelo, demostrando que aunque el tiempo invertido y la asistencia son pilares fundamentales, el éxito académico sigue siendo un fenómeno multifactorial.

# Paso 10

In [ ]:
import matplotlib.pyplot as plt

# Valores reales vs valores predichos para los dos modelos
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y_test, y_pred_lineal, alpha=0.6, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Valores reales')
axes[0].set_ylabel('Valores predichos')
axes[0].set_title('Regresión Lineal: Real vs. Predicho')

axes[1].scatter(y_test, y_pred_arbol, alpha=0.6, color='seagreen')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Valores reales')
axes[1].set_ylabel('Valores predichos')
axes[1].set_title('Árbol de Regresión: Real vs. Predicho')

plt.tight_layout()
plt.show()

# Identificamos el modelo con mejor desempeño según R²
mejor_modelo = df_resultados.loc[df_resultados['R²'].idxmax(), 'Modelo']
print(f"Modelo con mejor desempeño: {mejor_modelo}")

# Elegimos las predicciones del mejor modelo
if mejor_modelo == 'Regresión lineal':
    y_pred_mejor = y_pred_lineal
else:
    y_pred_mejor = y_pred_arbol

residuos = y_test - y_pred_mejor

# Análisis de residuos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuos vs valores predichos (para ver patrones/heterocedasticidad)
axes[0].scatter(y_pred_mejor, residuos, alpha=0.6, color='darkorange')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Valores predichos')
axes[0].set_ylabel('Residuos')
axes[0].set_title(f'Residuos vs. Predicciones ({mejor_modelo})')

# Distribución de los residuos
axes[1].hist(residuos, bins=20, color='darkorange', edgecolor='black')
axes[1].axvline(x=0, color='r', linestyle='--')
axes[1].set_xlabel('Residuo')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de los residuos')

plt.tight_layout()
plt.show()

print(f"Media de residuos: {residuos.mean():.4f}")
print(f"Desviación estándar de residuos: {residuos.std():.4f}")

Analisis de residuos

Los residuos se distribuyen de manera aproximadamente aleatoria alrededor de cero para 
la mayoría de las observaciones, sin un patrón sistemático evidente (no hay forma de 
embudo ni curvatura), lo que sugiere que el modelo captura razonablemente bien la 
relación lineal en el rango central de los datos (puntajes entre 60 y 75).

Sin embargo, se observa un grupo claro de valores atípicos: estudiantes con puntajes 
reales altos (80-98) para los cuales el modelo predice consistentemente valores mucho 
más bajos (60-73), generando residuos grandes (17-26). Esto se refleja en el histograma 
como una cola larga hacia la derecha, rompiendo la simetría de la distribución.

Esto indica que el modelo tiene dificultades específicas para predecir a los estudiantes 
de alto rendimiento, probablemente porque estos casos son poco frecuentes en los datos 
de entrenamiento y el modelo lineal tiende a "regresar hacia la media" en lugar de 
capturar esos extremos.

# Paso 11

In [ ]:
# --- Identificamos el mejor modelo (igual que en Paso 10) ---
if mejor_modelo == 'Regresión lineal':
    y_pred_mejor = y_pred_lineal
else:
    y_pred_mejor = y_pred_arbol

# --- Calculamos error absoluto por observación ---
errores_abs = np.abs(y_test - y_pred_mejor)

# Armamos un DataFrame con: valores reales, predichos, error, y las features originales
analisis = x_test.copy()
analisis['Valor_real'] = y_test.values
analisis['Valor_predicho'] = y_pred_mejor
analisis['Error_absoluto'] = errores_abs.values

# --- Casos con mayor error (peor desempeño) ---
peores_casos = analisis.sort_values('Error_absoluto', ascending=False).head(5)
print("Casos con MAYOR error absoluto:")
display(peores_casos)

# --- Casos con menor error (mejor desempeño) ---
mejores_casos = analisis.sort_values('Error_absoluto', ascending=True).head(5)
print("\nCasos con MENOR error absoluto (predicciones más precisas):")
display(mejores_casos)

# --- Comparación de promedios entre ambos grupos ---
print("\nPromedio de cada variable — peores casos vs mejores casos:")
comparacion = pd.DataFrame({
    'Peores casos (prom.)': peores_casos.select_dtypes(include=[np.number]).mean(),
    'Mejores casos (prom.)': mejores_casos.select_dtypes(include=[np.number]).mean()
})
display(comparacion)

## Análisis de casos con mayor y menor error — Regresión Lineal

Al comparar los 5 casos con mayor error absoluto (24-26 puntos) contra los 5 casos con 
menor error (prácticamente 0), se observa un patrón claro: **las variables disponibles 
(Attendance y Hours_Studied) son muy similares entre ambos grupos** (79.8% vs 76.8% de 
asistencia; 21.8 vs 17.2 horas de estudio), mientras que el valor real del examen es 
radicalmente distinto (92.8 vs 65.8 en promedio).

Esto indica que el modelo no está confundiéndose por variabilidad en las features que 
conoce, sino que **las variables Attendance y Hours_Studied no son suficientes para 
explicar los puntajes excepcionalmente altos** (>85). Estudiantes con niveles de 
asistencia y estudio comparables a la media terminan obteniendo resultados muy distintos, 
lo que sugiere la influencia de otros factores no capturados en el dataset actual — por 
ejemplo, conocimientos previos, calidad del estudio (no solo cantidad de horas), apoyo 
familiar, condiciones socioeconómicas, motivación, o características individuales del 
estudiante.

En términos de limitaciones: 
- **No es un problema del tipo de modelo** — un modelo lineal simple no podría capturar 
  esta relación aunque fuera más complejo, porque la señal simplemente no está en los 
  datos disponibles.
- **Es una limitación de las variables disponibles**: el dataset carece de features que 
  expliquen el desempeño excepcional, generando un "techo" de precisión para los casos 
  extremos.
- Una posible mejora futura sería incorporar variables adicionales (calificaciones 
  previas, tipo de escuela, acceso a recursos, etc.) que ayuden a diferenciar mejor a 
  los estudiantes de alto rendimiento.

# Paso 12

In [ ]:
from sklearn.model_selection import KFold, cross_validate
import numpy as np
import pandas as pd

# Definimos 5 particiones (k=5) con barajado
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Métricas a evaluar en las 5 particiones
scoring = {
    'MAE': 'neg_mean_absolute_error',
    'RMSE': 'neg_root_mean_squared_error',
    'R2': 'r2'
}

# Aplicamos validación cruzada sobre los datos de entrenamiento
cv_results = cross_validate(
    pipeline_lineal, 
    x_train, 
    y_train, 
    cv=kfold, 
    scoring=scoring, 
    return_train_score=False
)

# Convertimos los errores negativos a positivos
mae_cv = -cv_results['test_MAE']
rmse_cv = -cv_results['test_RMSE']
r2_cv = cv_results['test_R2']

# Construimos la tabla comparativa solicitada
comparativa_cv = pd.DataFrame({
    'Métrica': ['MAE', 'RMSE', 'R²'],
    'Evaluación en Prueba (Paso 8)': [mae_lin, rmse_lin, r2_lin],
    'Media en Validación Cruzada (k=5)': [round(mae_cv.mean(), 4), round(rmse_cv.mean(), 4), round(r2_cv.mean(), 4)],
    'Desviación Estándar (± std)': [round(mae_cv.std(), 4), round(rmse_cv.std(), 4), round(r2_cv.std(), 4)]
})

print("Comparación: Partición Simple (Paso 8) vs. Validación Cruzada (k=5):")
display(comparativa_cv)

# Desglose por fold
folds_df = pd.DataFrame({
    f'Partición {i+1}': [round(mae_cv[i], 4), round(rmse_cv[i], 4), round(r2_cv[i], 4)]
    for i in range(5)
}, index=['MAE', 'RMSE', 'R²'])

print("\nResultados individuales por partición:")
display(folds_df)

# Análisis y Comparación: Validación Cruzada (5 Particiones) vs. Partición Simple (Paso 8)

**1. Comparación de Resultados**
Al contrastar la evaluación obtenida en la partición simple de prueba (Paso 8) contra la media de la validación cruzada de 5 particiones sobre el conjunto de entrenamiento:
* **MAE:** La media obtenida en validación cruzada fue de **1.5469** con una desviación estándar de **$\pm 0.0478$**, frente a **1.4690** del conjunto de prueba.
* **RMSE:** La media en validación cruzada se ubicó en **2.6836** con una desviación estándar de **$\pm 0.1476$**, en comparación con **2.4102** en la prueba individual.
* **$R^2$:** La media del coeficiente de determinación fue de **0.5310** (53.1% de varianza explicada) con una desviación estándar de **$\pm 0.0254$**, frente al **0.5890** (58.9%) obtenido en la partición de prueba única.

**2. ¿Por qué la Validación Cruzada ofrece una referencia más estable?**
* **Corrección del sesgo de una sola división:** La partición simple de prueba arrojó un resultado ligeramente más optimista ($R^2 = 0.5890$). La validación cruzada revela que el desempeño promedio real ronda el 53.1%, evitando sobreestimar la capacidad del modelo por haber tenido una división de datos favorable por azar.
* **Medición de la dispersión (Desviación Estándar):** Al evaluar en 5 particiones distintas, no solo obtenemos un promedio, sino que comprobamos que la dispersión entre folds es muy baja ($\pm 0.0478$ en MAE), lo que confirma que el modelo se comporta de manera consistente y sin inestabilidad grave.
* **Prevención de fuga de información (*Data Leakage*):** Al ejecutar el `Pipeline` dentro de `cross_validate`, el preprocesamiento (escalado e imputación) se ajusta exclusivamente con los datos de entrenamiento de cada partición, garantizando una evaluación limpia y sin contaminación de datos.

# Paso 13: Conclusión General

**Modelo con mejor desempeño y mejora frente a la línea base**

La **Regresión Lineal** fue el modelo superior: redujo el MAE a **1.47 en prueba y 1.55 en CV** (más del 50% de mejora frente al modelo base) y alcanzó un **$R^2$ de 0.5890 en test set y 0.5310 ($\pm 0.0254$) en validación cruzada**, demostrando estabilidad y descartando sobreajuste severo.

**Variables más relevantes**
* **`Attendance` (Asistencia):** Factor con mayor impacto directo en la calificación.
* **`Hours_Studied` (Horas de Estudio):** Predictor complementario clave de relación positiva.

**Limitaciones identificadas**
* **Subestimación en puntajes altos (>85):** El modelo tiende a predecir hacia la media en estudiantes sobresalientes.
* **Señal insuficiente:** Alumnos con asistencia y horas de estudio idénticas obtienen calificaciones muy dispares, evidenciando que dos variables no bastan para explicar el éxito académico.

**Propuestas de mejora**
* Reincorporar variables del dataset como `Previous_Scores`, `Parental_Involvement` y `Tutoring_Sessions`.
* Probar modelos de ensamble no lineales (*Random Forest* o *XGBoost*) e ingeniería de características.

**Viabilidad en decisiones académicas**

El modelo es **suficiente como una primera aproximación de alerta temprana** para alumnos en riesgo por baja asistencia o estudio, pero **requiere mayor validación y variables adicionales antes de respaldar decisiones de alto impacto** (como becas o admisiones).
* **Omisión de dimensiones cualitativas y de contexto:** La calidad de la enseñanza, la efectividad del método de estudio, el conocimiento previo y las condiciones socioeconómicas no quedaron reflejadas al usar únicamente dos variables cuantitativas.

 
**Propuestas de Mejora para Etapas Posteriores**

* **Incorporar variables complementarias:** Reintegrar predictores como `Previous_Scores` (calificaciones previas), `Parental_Involvement`, `Access_to_Resources` y `Tutoring_Sessions` mediante codificación adecuada e ingeniería de características (*feature engineering*).
* **Modelos no lineales avanzados:** Explorar algoritmos de ensamble como *Random Forest* o *Gradient Boosting* (XGBoost / LightGBM) para capturar interacciones complejas entre variables sin perder generalización.
* **Técnicas de regularización:** Probar *Ridge* o *Lasso* si se amplía el espacio de variables para evitar multicolinealidad.

**Viabilidad para la Toma de Decisiones Académicas**

El modelo actual **se considera suficiente como una primera aproximación descriptiva y de diagnóstico temprano** (por ejemplo, para alertar sobre estudiantes en riesgo por baja asistencia o pocas horas de estudio). 

Sin embargo, **requiere mayor validación y la incorporación de más variables contextuales antes de emplearse para respaldar decisiones académicas de alto impacto** (como asignación de becas, planes de intervención especializada o admisiones), ya que su margen de error en casos sobresalientes o atípicos podría derivar en decisiones sesgadas o imprecisas.